# 02. Feature Engineering — 정제와 피처 계산을 함수로 만들기

**목적**
- `01_EDA`에서 정한 정제 규칙과 피처 계산을 함수로 옮긴다.
- 함수의 결과가 EDA에서 확인한 값과 일치하는지 검증한다.
- 검증이 끝난 함수는 `src/`로 옮겨 모델링에서 그대로 쓴다.

## 환경 설정

In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import medfilt

warnings.filterwarnings('ignore')

# 재현성
SEED = 42
np.random.seed(SEED)

# 경로
PROC_DIR = '../data/processed'
FIG_DIR = '../results/figures'

# 그래프 설정 (한글 폰트 포함)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. 데이터 불러오기

**목적**: `01_EDA`에서 `.mat` 파일을 읽어 저장해 둔 두 파일을 불러온다.
- `summary.parquet`: 사이클별 요약값 (방전 용량, 내부 저항, 온도, 충전 시간)
- `qdlin.npz`: 셀별 초기 101개 사이클의 방전 곡선 `Qdlin`과 전압 축 `Vdlin`

In [3]:
df = pd.read_parquet(os.path.join(PROC_DIR, 'summary.parquet'))

z = np.load(os.path.join(PROC_DIR, 'qdlin.npz'))
vdlin = z['Vdlin']
qdlin = {k: z[k] for k in z.files if k != 'Vdlin'}

print(df.shape, df['cell_id'].nunique())
print(len(qdlin), qdlin['b1c0'].shape)

(116722, 12) 139
139 (101, 1000)


## 2. 셀 선별

**목적**: 수명 값이 실제 EOL 도달 시점인 셀만 고른다.

**규칙 (EDA Q2-1 ~ Q2-3)**
- 방전 용량이 0.8~1.15Ah 안에 있는 행만 정상으로 본다.
- 정상 행 중 마지막 방전 용량이 0.885Ah 이하이면 EOL에 도달한 것으로 본다.
- 수명 값이 있고 EOL에 도달한 셀만 사용한다.

**기대하는 결과**: Batch 1 / 2 / 3에서 36 / 39 / 44셀.

In [4]:
QD_MIN, QD_MAX = 0.8, 1.15   # 방전 용량의 정상 범위 (Ah)
EOL_CHECK = 0.885            # 마지막 용량이 이 값 이하이면 EOL 도달


def select_cells(df):
    """셀 단위 표를 만들고 분석 대상 여부(use)를 붙인다."""
    cells = (df.drop_duplicates('cell_id')
             [['batch', 'cell_id', 'cycle_life', 'charging_policy']]
             .reset_index(drop=True))

    ok = df[df['QD'].between(QD_MIN, QD_MAX)]
    last = (ok.groupby('cell_id')
            .agg(n_cycles=('cycle', 'max'), qd_last=('QD', 'last'))
            .reset_index())
    cells = cells.merge(last, on='cell_id')

    cells['reached_eol'] = cells['qd_last'] <= EOL_CHECK
    cells['use'] = cells['reached_eol'] & cells['cycle_life'].notna()

    # 집단 구분: Batch 2는 newstructure 여부로 나눈다
    ns = cells['charging_policy'].str.contains('newstructure')
    cells['group'] = 'Batch ' + cells['batch'].astype(str)
    cells.loc[(cells['batch'] == 2) & ns, 'group'] = 'Batch 2 newstructure'
    cells.loc[(cells['batch'] == 2) & ~ns, 'group'] = 'Batch 2 일반'
    cells['policy'] = cells['charging_policy'].str.replace('-newstructure', '', regex=False)
    return cells


cells = select_cells(df)

print(cells.groupby('batch')['use'].agg(사용='sum', 전체='size'))
print()
print(cells[cells['use']].groupby('group')['cycle_life'].agg(['size', 'min', 'median', 'max']))

       사용  전체
batch        
1      36  46
2      39  47
3      44  46

                      size    min  median     max
group                                            
Batch 1                 36  534.0   772.5  1074.0
Batch 2 newstructure     9  777.0   904.0  1186.0
Batch 2 일반              30  392.0   451.0   514.0
Batch 3                 44  541.0  1005.5  1935.0


**결과 해석 — 셀 선별**

- 분석 대상이 36 / 39 / 44셀로 EDA(Q2-3)와 같다. 제외된 셀은 Batch 1의 10셀(EOL 미도달), Batch 2의 8셀(수명 값 없음), Batch 3의 2셀(EOL 미도달)이다.
- 집단 구분도 EDA와 같다. Batch 2는 일반 30셀(392~514)과 `newstructure` 9셀(777~1,186)로 나뉜다.
- EDA에서는 `newstructure` 구분과 집단 이름을 여러 단계에 걸쳐 붙였지만, 여기서는 `select_cells` 한 함수에서 만든다. 결과가 같으므로 이후에는 이 함수의 `use`, `group`, `policy` 열을 그대로 쓴다.

## 3. 방전 용량 정제

**목적**: 분석 대상 셀의 사이클별 기록에서 측정 오류를 제거한다.

**규칙 (EDA Q2-1, Q2-5)**
- 절대 범위: 0.8~1.15Ah 밖의 행을 제외한다.
- 주변 값 비교: 주변 11사이클의 중앙값과 0.04Ah 넘게 차이 나는 행을 제외한다.

**기대하는 결과**: 주변 값 비교로 제외되는 행이 11개.

In [5]:
DEV_WINDOW = 11   # 이동 중앙값을 구할 사이클 수
DEV_MAX = 0.04    # 이동 중앙값과의 차이 허용 한계 (Ah)


def clean_qd(df, cells):
    """분석 대상 셀의 사이클별 기록에서 방전 용량 이상값을 제거한다."""
    use_ids = cells.loc[cells['use'], 'cell_id']
    d = df[df['cell_id'].isin(use_ids) & df['QD'].between(QD_MIN, QD_MAX)]
    d = d.sort_values(['cell_id', 'cycle']).copy()

    med = d.groupby('cell_id')['QD'].transform(
        lambda x: x.rolling(DEV_WINDOW, center=True, min_periods=1).median())
    spike = (d['QD'] - med).abs() > DEV_MAX
    return d[~spike].copy(), int(spike.sum())


d_clean, n_spike = clean_qd(df, cells)

print('전체에서 절대 범위 밖인 행:', int((~df['QD'].between(QD_MIN, QD_MAX)).sum()))
print('주변 값 비교로 제외한 행:', n_spike)
print('정제 후:', d_clean.shape, '셀', d_clean['cell_id'].nunique())

전체에서 절대 범위 밖인 행: 59
주변 값 비교로 제외한 행: 11
정제 후: (97914, 12) 셀 119


**결과 해석 — 방전 용량 정제**

- 절대 범위로 59행, 주변 값 비교로 11행이 제외되어 EDA(Q2-1, Q2-5)와 같다.
- 정제 후 119셀, 97,914행이 남았다.
- 두 기준값(0.8~1.15Ah, 0.04Ah)을 함수 밖의 상수로 두었다. 기준을 바꿀 일이 생기면 한 곳만 고치면 된다.

## 4. ΔQ(V) 피처

**목적**: 핵심 피처 `log_var` = log10 Var(Q₁₀₀ − Q₁₀)를 계산한다.

**규칙 (EDA Q3-2 ~ Q3-5)**
- 사이클은 각 셀에서 처음 값이 있는 위치부터 센 측정 순서를 쓴다. Batch 1은 첫 사이클의 기록이 비어 있어 한 칸 밀린다.
- 100번째와 10번째 측정 사이클의 `Qdlin`을 뺀다.
- 전압 축 9포인트 중앙값 필터로 한 점짜리 스파이크를 제거한다.
- 곡선의 분산에 log10을 취한다.

**기대하는 결과**: Batch 1의 직선이 `log_life = −0.300 × log_var + 1.752`, 상관이 −0.84.

In [6]:
K_EARLY, K_LATE = 10, 100   # ΔQ에 쓰는 두 사이클 (측정 순서)
MEDFILT_SIZE = 9            # 전압 축 중앙값 필터의 크기


def first_valid_index(qdlin):
    """셀별로 Qdlin에 처음 값이 있는 위치(0부터)를 구한다."""
    return pd.Series({cid: int(np.argmax(~np.isnan(arr).all(axis=1)))
                      for cid, arr in qdlin.items()})


def delta_q_features(qdlin, cells, first_valid):
    """분석 대상 셀의 ΔQ(V) 곡선에서 log_var를 계산한다."""
    rows = []
    for cid in cells.loc[cells['use'], 'cell_id']:
        arr, off = qdlin[cid], first_valid[cid]
        dq = arr[off + K_LATE - 1] - arr[off + K_EARLY - 1]
        dq = medfilt(dq.astype(float), MEDFILT_SIZE)
        rows.append({'cell_id': cid, 'dq_var': np.var(dq), 'log_var': np.log10(np.var(dq))})
    return pd.DataFrame(rows)


first_valid = first_valid_index(qdlin)
feat = cells.loc[cells['use'], ['cell_id', 'batch', 'group', 'policy', 'cycle_life']].merge(
    delta_q_features(qdlin, cells, first_valid), on='cell_id')
feat['log_life'] = np.log10(feat['cycle_life'])

print('처음 값이 있는 위치 (배치별 최소, 최대)')
print(first_valid.groupby(first_valid.index.str[1]).agg(['min', 'max']))
print()
print('dq_var 범위:', feat['dq_var'].min().round(6), '~', feat['dq_var'].max().round(6))
print()
print('집단별 log_var와 log_life의 상관')
print(feat.groupby('group').apply(lambda g: g['log_var'].corr(g['log_life'])).round(2))

b1 = feat[feat['batch'] == 1]
slope, intercept = np.polyfit(b1['log_var'], b1['log_life'], 1)
print()
print(f'Batch 1 직선: log_life = {slope:.3f} * log_var + {intercept:.3f}')

처음 값이 있는 위치 (배치별 최소, 최대)
   min  max
1    1    1
2    0    0
3    0    0

dq_var 범위: 7e-06 ~ 0.000821

집단별 log_var와 log_life의 상관
group
Batch 1                -0.84
Batch 2 newstructure   -0.27
Batch 2 일반             -0.38
Batch 3                -0.76
dtype: float64

Batch 1 직선: log_life = -0.300 * log_var + 1.752


**결과 해석 — ΔQ(V) 피처**

- 사이클 기준이 의도대로 적용됐다. Batch 1은 46셀 모두 한 칸 밀려 있고 Batch 2·3은 밀림이 없다.
- `log_var`가 EDA(Q3-5, Q3-6)와 같은 값으로 계산됐다. 분산의 범위, 집단별 상관, Batch 1의 직선이 모두 일치한다.
- Batch 2 `newstructure`의 상관(−0.27)은 9셀로 계산한 값이어서 참고만 한다.
- 이 함수는 수명 값을 쓰지 않는다. 입력이 초기 100사이클의 방전 곡선뿐이므로 평가 배치에도 같은 함수를 그대로 적용할 수 있다.

## 5. 용량 기울기 피처

**목적**: 두 번째 후보 피처 `qd_slope_last`를 계산한다. 91~100번째 측정 사이클의 방전 용량에 직선을 맞춘 기울기다.

**규칙 (EDA Q5-1)**
- 정제된 방전 용량(`d_clean`)을 쓴다.
- 사이클은 `log_var`와 같은 측정 순서 기준을 쓴다.

**기대하는 결과**: `log_life`와의 상관이 +0.50 / +0.26 / +0.38, Batch 1에서 `log_var`와의 상관이 −0.50.

In [7]:
SLOPE_FROM, SLOPE_TO = 91, 100   # 기울기를 구하는 측정 순서 구간


def qd_slope_features(d_clean, first_valid):
    """정제된 방전 용량에서 91~100번째 측정 사이클의 기울기를 계산한다."""
    e = d_clean.copy()
    e['k'] = e['cycle'] - e['cell_id'].map(first_valid)   # 측정 순서 (1부터)
    e = e[e['k'].between(SLOPE_FROM, SLOPE_TO)]

    rows = []
    for cid, g in e.groupby('cell_id'):
        rows.append({'cell_id': cid,
                     'qd_slope_last': np.polyfit(g['k'], g['QD'], 1)[0],
                     'n_slope': len(g)})
    return pd.DataFrame(rows)


feat = feat.drop(columns=['qd_slope_last', 'n_slope'], errors='ignore').merge(
    qd_slope_features(d_clean, first_valid), on='cell_id')

print(feat.shape, '결측:', int(feat['qd_slope_last'].isna().sum()))
print('기울기 계산에 쓴 사이클 수:', feat['n_slope'].value_counts().to_dict())
print()
print('집단별 qd_slope_last와 log_life의 상관')
print(feat.groupby('group').apply(lambda g: g['qd_slope_last'].corr(g['log_life'])).round(2))
print()
b1 = feat[feat['batch'] == 1]
print('Batch 1에서 log_var와의 상관:', round(b1['qd_slope_last'].corr(b1['log_var']), 2))

(119, 10) 결측: 0
기울기 계산에 쓴 사이클 수: {10: 119}

집단별 qd_slope_last와 log_life의 상관
group
Batch 1                 0.50
Batch 2 newstructure   -0.60
Batch 2 일반              0.26
Batch 3                 0.38
dtype: float64

Batch 1에서 log_var와의 상관: -0.5


**결과 해석 — 용량 기울기 피처**

- `qd_slope_last`가 EDA(Q5-2)와 같은 값으로 계산됐다. 119셀 모두 10개 사이클로 기울기를 구했고 결측이 없다.
- Batch 1에서 `log_var`와의 상관이 −0.50으로 EDA와 같다. 두 피처는 절반쯤 겹친다.
- **Batch 2 `newstructure` 9셀에서는 상관이 −0.60으로 부호가 반대다.** EDA의 부호 일치 판정은 세 집단(Batch 1, Batch 2 일반, Batch 3)으로만 했고 이 집단은 포함하지 않았다.
- 9셀로 계산한 값이어서 관계가 반대라고 단정할 수는 없다. 다만 이 피처를 후보로 남긴 근거(부호의 일관성)가 약해졌다.
- 계획은 그대로 둔다. 추가 여부는 Batch 1 교차검증으로 정하고, 평가 단계에서 이 피처를 넣은 모델의 `newstructure` 집단 오차를 따로 확인한다.

## 6. `src/`로 옮긴 함수 검증

**목적**: 위에서 검증한 함수를 `src/preprocess.py`, `src/features.py`로 옮겼다. `src`의 `build_features`가 이 노트북에서 만든 피처 표와 같은 값을 내는지 확인한다.

**기대하는 결과**: 119셀의 `log_var`, `qd_slope_last`가 완전히 일치한다.

In [8]:
import sys
sys.path.append('..')   # 저장소 최상위를 경로에 추가

from src.preprocess import load_cache
from src.features import build_features

df_s, qdlin_s, vdlin_s = load_cache(PROC_DIR)
feat_src = build_features(df_s, qdlin_s)

a = feat.sort_values('cell_id').reset_index(drop=True)
b = feat_src.sort_values('cell_id').reset_index(drop=True)

print('셀 수:', len(a), len(b))
print('셀 목록 일치:', (a['cell_id'] == b['cell_id']).all())
for col in ['log_var', 'qd_slope_last', 'log_life']:
    print(f'{col} 최대 차이:', (a[col] - b[col]).abs().max())
print('집단 일치:', (a['group'] == b['group']).all())

셀 수: 119 119
셀 목록 일치: True
log_var 최대 차이: 0.0
qd_slope_last 최대 차이: 0.0
log_life 최대 차이: 0.0
집단 일치: True
